在前面的 ZeRO 三阶段中，我们通过多卡分片将单卡模型状态压缩到 $16\Psi/N$。但当 GPU 数量有限或模型过于庞大时，即使这个压缩后的值仍会撑爆 GPU 显存。**Offload 和 Infinity 正是在分片基础上，进一步将数据搬迁到 CPU 内存甚至 NVMe 固态硬盘，用“存储层级换显存”的终极方案。**

我们将系统地重构 **ZeRO-Offload** 与 **ZeRO-Infinity** 的完整技术细节，重点阐明之前提及的三个关键问题：

1.  **逻辑分片的实现**：在物理共享的 CPU 内存和 NVMe 上，如何通过软件约定实现类似 GPU 显存的“分片隔离”？
2.  **梯度裁剪的全局同步**：为什么多卡 Offload/Infinity 必须等待全局梯度裁剪，而单卡可以绕过？
3.  **GPU 与 CPU 的流水线重叠**：GPU 计算梯度、CPU 更新参数、下一轮前向这三者之间精确的时间线是怎样的？

---

## 一、ZeRO-Offload：将优化器状态卸载至 CPU 内存

### 1.1 动机与核心思想
在混合精度 + Adam 训练中，优化器状态（FP32 主参数、动量、方差）占据了 **$12\Psi$** 字节，是显存最大的单一开销。然而，Adam 更新仅为标量运算，完全可以在 CPU 上高效完成。Offload 的核心思想是：**将优化器状态和更新计算全部搬到 CPU 内存，GPU 只负责前向/反向的高强度矩阵计算，仅保留参数和梯度**。

Offload 通常与 **ZeRO Stage 2** 配合（也可配 Stage 3），在分片的基础上进一步降低 GPU 显存压力。

### 1.2 数据布局与显存公式
假设数据并行度为 $N$，使用 **ZeRO Stage 2 + Offload**，单卡的数据分布如下：

| 存储层级 | 数据内容 | 大小（每卡） |
|----------|----------|-------------|
| **GPU 显存** | FP16 参数（完整） | $2\Psi$ |
| | FP16 梯度（分片） | $\frac{2\Psi}{N}$ |
| **CPU 内存** | FP32 优化器状态（动量、方差、主参数，分片） | $\frac{12\Psi}{N}$ |

单卡 GPU 显存占用降至：
$$
M_{\text{GPU}} = 2\Psi + \frac{2\Psi}{N}
$$
以 $N=8$ 为例，$M_{\text{GPU}} \approx 2.25\Psi$。一个 13B 模型（$\Psi=13\times10^9$）仅需约 $2.25 \times 13\text{B} \times 2 \text{ bytes} \approx 58.5$ GB（实际小于此值，因参数为 FP16），单张 80GB A100 即可训练。相比纯 Stage 2（需约 $3.75\Psi$），省去了 $\frac{12\Psi}{N}$ 的优化器状态显存。

### 1.3 训练流程与通信
对于 GPU $k$（$k=0,\dots,N-1$），一次迭代步骤如下：

1.  **前向传播**（GPU）：使用完整 FP16 参数 $\theta^{\text{fp16}}$ 计算损失 $L_k$，并保留所有激活值（或按需使用检查点）。
2.  **反向传播**（GPU）：逐层计算局部梯度 $G_k^{(l)} = \nabla_{W_l} L_k$（完整尺寸，FP16）。
3.  **梯度同步**（GPU 间）：对每一层执行 **Reduce-Scatter**，得到属于本卡的分片全局平均梯度 $\bar{G}_k$（大小 $\frac{2\Psi}{N}$）。
    $$
    \bar{G}_k = \frac{1}{N}\sum_{j=0}^{N-1} G_j[k]
    $$
4.  **梯度卸载**（GPU → CPU）：将 $\bar{G}_k$ 转为 FP32，通过 PCIe 异步拷贝至 CPU 内存。
5.  **优化器更新**（CPU）：CPU 使用分片梯度更新对应的 Adam 状态（$m_k, v_k, \theta_k^{\text{fp32}}$），生成新的参数分片 $\theta_k^{\text{fp16}}$。
6.  **参数回传**（CPU → GPU）：更新后的 $\theta_k^{\text{fp16}}$ 通过 PCIe 拷回 GPU 显存。
7.  **参数重组**（GPU 间）：执行 **All-Gather**，将所有 GPU 的新参数分片拼成完整参数 $\theta^{\text{fp16}}$，供下一轮使用。

**通信量分析**：
- GPU 间通信：Reduce-Scatter ($2\Psi$) + All-Gather ($\Psi$) = $3\Psi$，与纯 Stage 2 完全一致。
- GPU ↔ CPU 搬运：每个 step 每卡搬运梯度分片 $\frac{2\Psi}{N}$ 字节（FP16）和参数分片 $\frac{2\Psi}{N}$ 字节（FP16），合计 **$\frac{4\Psi}{N}$ 字节**。由于分片，搬运量与模型总规模无关，仅占 $1/N$，极大减轻 PCIe 压力。

### 1.4 关键细节一：GPU 与 CPU 的流水线重叠（精确时间线）

Offload 的高效源于 **反向传播的倒序特性**，允许 GPU 继续算前几层时，CPU 已在后台更新后几层。

假设一个简单的 3 层模型（L3 为最后一层，L1 为最浅层）。以下是反向传播末尾到下一轮前向开始之间的流水线时间线：

| 时间点 | GPU 执行 | CPU 执行 |
|--------|----------|----------|
| **t1** | 反向算 L3 的局部梯度，Reduce-Scatter 得 g3，**立即发往 CPU** | 空闲 |
| **t2** | 反向算 L2 的局部梯度 | 收到 g3，**开始对 L3 执行 Adam 更新** |
| **t3** | 反向算 L1 的局部梯度，Reduce-Scatter 得 g1，发往 CPU | 完成 L3 更新，将新参数 L3' 传回 GPU；收到 g2，开始更新 L2 |
| **t4** | **等待全局梯度裁剪**（若多卡）或**立即开始前向**（若单卡，见下节） | 继续更新 L2，随后更新 L1 |
| **t5** | 前向算 L1（需要 L1 新参数） | 若 L1 已更新完则已就绪，否则 GPU 短暂等待 |
| **t6** | 前向算 L2 | L2 新参数已在 t4-t5 期间传回，直接可用 |
| **t7** | 前向算 L3 | L3 新参数早已就绪（t3 已传回） |

**关键观察**：
- GPU 在算 L2、L1 反向时，CPU 并行更新了 L3 和 L2。这段时间的重叠是“免费”的。
- 下一轮前向必须从 L1 开始，而 L1 是最后才开始更新的。因此 **GPU 仅需等待 L1 的更新**，而 L1 通常是最浅层、参数量较小，等待时间极短。
- L3 和 L2 的更新则完全被 GPU 后续计算和前向 L2/L3 的时间所掩盖。

**同步模式对比**：若无流水线，总时间为 $T_{\text{bwd}} + T_{\text{update}} + T_{\text{fwd}}$。而有流水线后，总时间 $\approx T_{\text{bwd}} + T_{\text{fwd}} + \max(0, T_{\text{update\_L1}} - T_{\text{其他}})$，更新延迟几乎被完全隐藏。

### 1.5 关键细节二：梯度裁剪的全局同步

全局梯度裁剪（Gradient Clipping）是大模型训练稳定性的重要手段。其数学公式要求先计算全模型所有层梯度的全局 L2 范数：
$$
\text{total\_norm} = \sqrt{\sum_{l=1}^{L} \|\nabla W_l\|^2}
$$
然后统一缩放所有梯度：
$$
\nabla W_l \leftarrow \nabla W_l \cdot \frac{\text{clip\_value}}{\max(\text{total\_norm}, \text{clip\_value})}
$$

**多卡 Offload 必须等待全局裁剪**：
- 在分布式环境下，每张卡只持有部分梯度分片。要计算 `total_norm`，必须等待所有卡的所有层梯度完成 Reduce-Scatter，并跨卡通信汇总范数平方（通常由 All-Reduce 实现）。
- 在所有梯度就绪之前，无法计算出全局缩放因子，也就无法更新任何参数。因此 **CPU 绝不能在梯度到齐之前提前更新某一层**，否则会使用错误的缩放量。
- 此外，混合精度的动态损失缩放也会在每步结束时检查全局梯度是否含有 NaN/Inf，若有一处溢出，整个 step 作废。提前更新将导致无法回滚。

**单卡 Offload 可以绕过**（极限流水线）：
- 单卡场景没有跨卡通信，所有梯度天然在本地。
- DeepSpeed 的单卡 Offload 模式可配置为 **分层梯度裁剪**（Layer-wise Clipping），即每层独立裁剪，无需等待全局范数。甚至可以在稳定训练时省略裁剪。
- 这使得单卡可以做到“反向算完一层就立刻把该层梯度发给 CPU 更新”，实现极致的逐层流水线，GPU 几乎无需等待。
- **代价**：牺牲了严格的全局裁剪一致性，但实践经验表明对收敛影响极小，是工程上划算的取舍。

### 1.6 显存与性能总结
- **显存节省**：GPU 显存降至 $2\Psi + \frac{2\Psi}{N}$，使单卡或少卡训练 10B+ 模型成为可能。
- **通信开销**：GPU 间通信不变，GPU↔CPU 搬运量为 $\frac{4\Psi}{N}$，与分片度成反比。
- **计算开销**：无额外计算，仅增加数据搬运和 CPU 更新（可与 GPU 计算重叠）。

---

## 二、ZeRO-Infinity：打通 GPU → CPU → NVMe 三级存储

### 2.1 动机：当 CPU 内存也不够时
Offload 将优化器状态卸载至 CPU，但若模型再大（如 500B～1T），单卡分片后的优化器状态仍可能超过 CPU 内存容量（例如 1T 模型，每卡 $\frac{12\text{TB}}{8}=1.5\text{TB}$，接近或超出高端服务器 2TB 内存上限）。此时必须将一部分数据进一步下沉到容量更大但速度更慢的 **NVMe 固态硬盘**。

ZeRO-Infinity 正是 **ZeRO Stage 3 分片 + 多级异构存储引擎** 的融合，将 GPU 显存、CPU 内存、NVMe 盘池化为一个统一的虚拟显存。

### 2.2 三级存储架构与“逻辑分片”的实现

物理上，一台多 GPU 服务器的 CPU 内存和 NVMe 是所有 GPU 共享的。Infinity 通过严格的软件约定实现了逻辑分片和物理亲和性绑定，避免 PCIe 总线争抢。

**实现逻辑分片的三种机制**：

1.  **Owner 规则**：每个参数分片、梯度分片、优化器状态分片都有一个“Owner GPU”。只有 Owner 有权从自己的本地存储中读取并更新该分片。其他 GPU 需要该分片时，必须通过 GPU 间高速网络（NVLink/InfiniBand）向 Owner 请求，绝不允许跨卡直接访问共享内存或 NVMe。
2.  **NUMA 亲和性绑定**：在多路 CPU 服务器上，DeepSpeed 会将 GPU $k$ 的分片物理分配在与其直连的 CPU 内存控制器上。尽管地址空间统一，但跨 NUMA 访问延迟极高，这从物理上加固了隔离。
3.  **NVMe 文件分区**：每张 GPU 在 NVMe 上拥有独立的文件或命名空间（如 `/nvme/gpu0_states.bin`），读写完全独立，文件系统保证互不干扰。

这种“物理共享、逻辑私有”的设计，使得每张 GPU 的 PCIe 通道仅服务于自己的数据搬运，8 卡可并行工作，避免了共享总线的瓶颈。

### 2.3 数据布局与单卡显存公式

Infinity 必须建立在 **ZeRO Stage 3** 之上（参数、梯度、优化器状态全部按 $N$ 分片）。单卡负责的总数据量为 $\frac{16\Psi}{N}$，但这部分数据不会全部驻留在某一层存储，而是按“热度”动态分布：

| 层级 | 介质 | 典型容量 | 带宽 | 存放内容（单卡视角） |
|------|------|----------|------|---------------------|
| **L0** | GPU HBM | 80 GB | ~2 TB/s | 当前层的完整参数（All-Gather 临时）、当前批次的激活值、工作缓冲区 |
| **L1** | CPU DRAM | 1-2 TB | 50-100 GB/s | 本卡负责的参数/梯度/优化器状态的热点页、预取缓冲区 |
| **L2** | NVMe SSD | 10-30 TB | 3-7 GB/s | 冷数据：暂时不用的优化器状态页、后续层的参数分片 |

GPU 显存中不保存任何持久化的模型状态分片，仅保留当前计算所需的工作集。因此单卡 GPU 峰值显存近似为：
$$
M_{\text{GPU}} \approx \max_l(\text{sizeof}(W_l) + \text{sizeof}(\text{activations}_l)) + \text{buffer}
$$
与总模型规模 $\Psi$ 几乎无关，而取决于层的最大尺寸。

### 2.4 训练流程与数据流

以 GPU $k$ 处理第 $l$ 层为例，完整一次迭代的微观过程如下。

#### 前向传播（逐层）
1.  **预取**：Infinity 调度器提前判断 Layer $l$ 的参数分片位置。若在 L2（NVMe），则发出异步读取指令，将分片从 NVMe 读至 L1 中的预取缓冲区；若已在 L1，直接使用。
2.  **All-Gather 参数**：所有 GPU 贡献自己拥有的 Layer $l$ 分片，拼出完整权重 $W_l$。注意，非 Owner 的分片是 Owner 从自己的 L1/L0 加载并通过 GPU 间网络传来的。
3.  **前向计算**：GPU 使用完整 $W_l$ 和本地输入 $A_{l-1}$ 计算 $Z_l, A_l$。
4.  **激活值卸载（可选）**：若激活检查点开启，$A_l$ 被异步写回 CPU 内存，然后立即在 GPU 中丢弃。
5.  **丢弃非本地分片**：释放 $W_l$ 中不属于本卡的分片，仅保留自己的 $\theta_{k,l}^{\text{fp16}}$（如果设计为常驻 GPU，否则亦可卸载）。

#### 反向传播（逐层，从 L 到 1）
1.  **预取参数**：若 $W_l$ 已不在 GPU（前向用完已丢弃），重新从 L1/L2 预取并 All-Gather。
2.  **重算激活值**（若开启检查点）：从 CPU 取回 $A_{l-1}$，重做本层前向得到 $A_l$。
3.  **计算局部梯度**：用上游梯度 $\delta_l$ 和 $A_l$ 计算对完整权重的局部梯度 $G_k^{(l)}$（FP16，大小 $\Psi_l$）。
4.  **Reduce-Scatter 梯度**：将 $G_k^{(l)}$ 分片，跨卡聚合后 GPU $k$ 仅保留自己的梯度分片 $\bar{G}_k^{(l)}$，其余丢弃。
5.  **梯度卸载**：将 $\bar{G}_k^{(l)}$ 异步写回 CPU 内存（L1），或直接写回 NVMe（L2），取决于调度策略。
6.  **释放临时参数与激活值**。

#### 优化器更新（全在 CPU，分页流水线）
反向传播结束后，所有梯度分片已位于 CPU/NVMe。优化器更新在 CPU 上以 **分页方式** 异步执行：

- Adam 状态被切分为固定大小的页（如 1MB），大部分在 NVMe 上，CPU 内存仅缓存正在更新的活跃页。
- 更新引擎按页顺序：
  1. 若该页的动量/方差在 NVMe，先异步读取至 CPU 内存。
  2. 执行 Adam 更新，得到新的参数页。
  3. 将新参数页标记为“就绪”，并根据需要写回 NVMe 或留在 CPU 供预取。
- 整个更新过程与 GPU 的下一轮计算流完全解耦：GPU 在算这一步时，CPU 在更新上一步的页，NVMe 在搬运冷页。

### 2.5 梯度裁剪的全局同步（同 Offload 多卡）
Infinity 同样依赖全局梯度裁剪。多卡环境下，必须等待所有层的梯度完成 Reduce-Scatter，并跨卡汇总 total_norm，才能计算缩放因子。因此，**CPU 更新必须在这个全局同步点之后才能启动**，无法像单卡那样逐层立即更新。

不过，Infinity 的优化器更新在同步点之后依然是高度流水线化的：一旦裁剪完成，CPU 立即开始更新，而 GPU 可以同时开始下一轮前向的预取（因为浅层新参数可能尚未就绪，但深层新参数和后续层旧参数的预取可以重叠）。

### 2.6 带宽感知调度与重叠条件

Infinity 要应对的最大挑战是 NVMe 极低的带宽（仅 3-7 GB/s）。为隐藏其延迟，引擎会：

- **静态分析**每一层的计算量 $T_{\text{comp}}^{(l)}$ 和参数量 $M_l$。
- 若 $T_{\text{comp}}^{(l)} \ge \frac{M_l}{B_{\text{NVMe}}}$，则可将该层参数的 NVMe 读取完全隐藏在计算中。
- 若计算太快（如 LayerNorm），则必须将该层参数常驻在 L1（CPU 内存）甚至 L0（GPU 显存），避免成为瓶颈。
- 动态切分：大层（如 FFN）被切成多个子块，交错预取与计算，实现细粒度重叠。

### 2.7 通信拓扑优化
- **本地 PCIe 独占**：每张 GPU 仅通过自己的 PCIe 链路访问其本地 CPU 内存/NVMe 分区。8 张卡提供 8 条独立通道，总带宽线性扩展。
- **跨卡数据走 GPU 间高速网络**：需要其他 GPU 的分片时，由 Owner GPU 从其本地存储加载到显存，再通过 NVLink/InfiniBand 传输。绝不通过 PCIe 跨卡读共享内存。
- 在无 NVLink 的消费级显卡（如 RTX 4090）上，跨卡通信被迫经由 PCIe 中转 CPU 内存，效率骤降，但分片逻辑依然是唯一能运行大模型的途径。

### 2.8 Offload 与 Infinity 的对比

| 特性 | ZeRO-Offload | ZeRO-Infinity |
|------|-------------|---------------|
| **搭配的 ZeRO 阶段** | Stage 2（或 Stage 3） | 必须 Stage 3 |
| **卸载介质** | CPU 内存 | CPU 内存 + NVMe SSD |
| **单卡 GPU 显存占用** | $2\Psi + \frac{2\Psi}{N}$ | 与模型规模无关，仅当前工作集 |
| **逻辑分片方式** | Owner 规则 + NUMA 绑定 | Owner 规则 + NUMA 绑定 + NVMe 文件分区 |
| **GPU↔CPU 搬运量/step** | $\frac{4\Psi}{N}$（梯度+参数分片） | 随层动态变化，仅搬运所需页 |
| **梯度裁剪同步** | 多卡必须等，单卡可绕开 | 多卡必须等 |
| **流水线重叠机制** | 反向倒序 + 前向正序的错位 | 多级预取 + 分页异步更新 |
| **适用模型规模** | 单卡数十亿，多卡百亿～千亿 | 多卡千亿～万亿 |
| **工程复杂度** | 中等 | 极高（自研内存分配、磁盘缓存、分页调度） |

---

## 三、总结

- **Offload** 解决了 GPU 显存放不下优化器状态的问题，通过将 $12\Psi/N$ 的 Adam 状态卸载至 CPU 内存，并利用反向传播倒序与下一轮前向的错位，将 CPU 更新延迟几乎完全隐藏。单卡模式可进一步绕过全局梯度裁剪实现逐层极致流水线。
- **Infinity** 则在 CPU 内存也不够时，引入 NVMe 作为第三级存储。它在 ZeRO Stage 3 分片的基础上，通过 Owner 规则、NUMA 绑定、文件分区实现逻辑隔离，用带宽感知调度和分页预取掩盖硬盘的低速，最终让万亿参数模型训练成为可能。
- 两者的核心设计哲学一脉相承：**用逻辑分片消除冗余，用异构存储扩展容量，用流水线重叠隐藏延迟，用少量计算换极致显存**。